# Padronizando colunas e corrigindo tipos

## Task 2: padronizar os nomes das colunas e corrigir os tipos

## O problema

- O público chega como texto com vírgula de milhar (`"62,576"`), e a data num formato próprio
  (`"13 Jun 1950 - 12:00"`): não dá para somar nem filtrar por período.
- As colunas têm espaços (`Home Team Name`).

Objetivo: nomes simples, em português, e cada coluna com o tipo certo.

## select: escolher e renomear colunas

`select` monta um DataFrame só com as colunas pedidas; `.alias()` dá o nome novo.

`F.col("ano")`: uma **referência a uma coluna**, avaliada só na execução. Aceita nomes com espaço.

## filter: condições compostas

`filter` e `where` são a mesma função. Para combinar: `&` (e), `|` (ou), `~` (não), com **cada condição entre
parênteses**.

Sem parênteses, o `&` é calculado antes do `>=` (prioridade do Python). `and`, `or` e `not` não funcionam com colunas.

## withColumn: corrigindo o público

`withColumn("nome", expressão)` cria uma coluna, ou substitui uma existente. Primeiro, a conversão direta com `cast`:

Erro: com o **modo ANSI**, padrão no Spark 4, um `cast` inválido para o job.

- **`cast`**: converte e **lança erro** se o valor for inválido.
- **`try_cast`**: converte e devolve **`null`** se o valor for inválido. Depois, é preciso contar os nulos.

## withColumn: corrigindo a data

Formato `"13 Jun 1950 - 12:00"` = `"dd MMM yyyy - HH:mm"`. Converte com `to_timestamp`; a versão segura é
`try_to_timestamp`. Teste com uma data boa e uma ruim:

`to_timestamp` lança erro com a data inválida; `try_to_timestamp` devolve `null`. O formato vai dentro de `F.lit(...)`.

## Juntando tudo: a tabela padronizada

Tipos corrigidos e duas colunas novas: `total_gols` e `resultado` (`F.when(...).otherwise(...)`, o `CASE WHEN` do SQL).

## Conferindo a conversão: conte os nulos

Depois de todo `try_cast` ou `try_to_timestamp`, **conte os nulos**: `isNull()` vira 1 ou 0, e a soma dá o total.

## Resultado da task 2

Nomes padronizados, público como número, data como data e hora, e nenhum nulo novo. Próxima task: as estatísticas.

## Para consultar depois

- **Três jeitos de se referir a uma coluna:** `"ano"`, `F.col("ano")` e `jogos.ano`. Os três funcionam; o `F.col` é o
  mais robusto.
- **Outros filtros:** `F.col("mandante").isin("Brazil", "Argentina")` (está na lista) e
  `F.col("fase").contains("final")` (contém o texto). O `contains` diferencia maiúsculas de minúsculas: para ignorar,
  use `F.lower(F.col("fase")).contains("final")`.
- **`cast` x `try_cast`:** com o modo ANSI (padrão no Spark 4), `cast` para o job diante de um valor inválido, e
  `try_cast` devolve `null`. O mesmo vale para `to_timestamp` e `try_to_timestamp`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()